# 03 — Target Feasibility Analysis

## Purpose

Determine whether the current full DOT NYC 311 API extract can support
the proposed binary target `missed_resolution_target`.

The notebook evaluates:

- whether the target-required fields exist;
- whether the fields contain usable values;
- whether a valid historical eligible population exists;
- whether the proposed target can be constructed without inventing labels;
- whether target-related fields create prediction-time leakage risks.

This notebook does not train a model and does not select an alternative agency.
A negative feasibility finding is a valid professional outcome: it prevents
an unsupported label from entering later modelling work.


## 2. Business target definition

**Target name:** `missed_resolution_target`

- `1` = complaint closed after its expected due date.
- `0` = complaint closed on or before its expected due date.

The theoretical definition is:

```python
eligible_for_target = closed_date.notna() & due_date.notna()
missed_resolution_target = (closed_date > due_date).astype("int8")
```

`closed_date` is the actual closure timestamp and `due_date` is the expected
deadline; both are required. `created_date` supplies temporal context and
validates population consistency. `status` supports lifecycle interpretation
but is not the target.


## 3. Feasibility criteria

- **Feasible for construction:** at least one valid historical record contains
  all timestamps required to evaluate the candidate target.
- **Not feasible:** no record contains the required timestamps, or their
  comparison cannot be evaluated reliably.

Day 4 uses the zero-eligible-record rule. A non-zero eligible population is
not automatically production-ready; class balance, status exclusions,
deadline semantics, and prediction-time availability still require review.


## 4. Imports


In [1]:
from datetime import datetime, timezone
from http.client import HTTPSConnection, RemoteDisconnected
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urlsplit
from urllib.request import Request, urlopen
import json
import os
import socket
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 160)


## 5. Project path and configuration

Paths are resolved by searching upward for the repository's `src/urban_ops`
package, matching the established notebook convention. Outputs stay inside
the repository. The current repository uses `requirements.txt` and does not
yet contain the planned `pyproject.toml`.


In [2]:
PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "src" / "urban_ops").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from urban_ops.data.nyc_311_config import (
    API_ENDPOINT,
    API_MAX_ATTEMPTS,
    API_TIMEOUT_SECONDS,
)

REPORT_TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
TARGET_DEFINITION_DRAFT_PATH = PROJECT_ROOT / "docs" / "target_definition_draft.md"
REPORT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_NAME = "missed_resolution_target"
TARGET_REQUIRED_COLUMNS = [
    "unique_key",
    "created_date",
    "closed_date",
    "due_date",
    "status",
]
TARGET_DATE_COLUMNS = ["created_date", "closed_date", "due_date"]
TARGET_COLUMN_PURPOSES = {
    "unique_key": "Complaint identifier",
    "created_date": "Creation timestamp and temporal context",
    "closed_date": "Actual complaint closure timestamp",
    "due_date": "Expected resolution deadline",
    "status": "Supports eligibility and lifecycle interpretation",
}

PROJECT_ROOT, REPORT_TABLES_DIR.relative_to(PROJECT_ROOT)


(PosixPath('/Users/mohammadmubashir/VCode/urban-operations-intelligence-platform'),
 PosixPath('reports/tables'))

## 6. API extraction configuration

The source is the official NYC Open Data endpoint. The configured scope is
DOT records from the beginning of the 2020-to-present dataset through a
snapshot upper bound captured at extraction time. An optional Socrata
application token is read from `NYC_OPEN_DATA_APP_TOKEN`; its value is never
displayed or persisted.


In [3]:
AGENCY = "DOT"
CONFIGURED_START_DATE = "2020-01-01T00:00:00.000"
CONFIGURED_END_DATE: str | None = None
API_PAGE_SIZE = 50_000
API_RETRY_BACKOFF_SECONDS = 2
API_ORDERING = "unique_key ASC"
EXTRACTION_COLUMNS = [
    "unique_key",
    "created_date",
    "closed_date",
    "due_date",
    "agency",
    "agency_name",
    "complaint_type",
    "status",
    "resolution_description",
    "resolution_action_updated_date",
]
DATASET_ID = API_ENDPOINT.rsplit("/", maxsplit=1)[-1].removesuffix(".json")
DATASET_METADATA_URL = f"https://data.cityofnewyork.us/api/views/{DATASET_ID}"
APP_TOKEN_ENVIRONMENT_VARIABLE = "NYC_OPEN_DATA_APP_TOKEN"
APP_TOKEN = os.getenv(APP_TOKEN_ENVIRONMENT_VARIABLE)

api_configuration = pd.DataFrame(
    [
        {
            "api_endpoint": API_ENDPOINT,
            "dataset_id": DATASET_ID,
            "agency": AGENCY,
            "configured_start_date": CONFIGURED_START_DATE,
            "configured_end_date": CONFIGURED_END_DATE or "live snapshot",
            "page_size": API_PAGE_SIZE,
            "ordering": API_ORDERING,
            "timeout_seconds": API_TIMEOUT_SECONDS,
            "maximum_attempts": API_MAX_ATTEMPTS,
            "app_token_configured": bool(APP_TOKEN),
        }
    ]
)
display(api_configuration.T)


,0
api_endpoint,https://data.cityofnewyork.us/resource/erm2-nwe9.json
dataset_id,erm2-nwe9
agency,DOT
configured_start_date,2020-01-01T00:00:00.000
configured_end_date,live snapshot
page_size,50000
ordering,unique_key ASC
timeout_seconds,180
maximum_attempts,3
app_token_configured,False


## 7. Full DOT API loading

The request helper reuses the bounded retry and TLS-verified DNS fallback
pattern established by the first two notebooks. The loader first captures a
source maximum `created_date`, then retrieves every DOT row inside that fixed
snapshot boundary with deterministic pagination. It validates the final row
count and identifier uniqueness so a partial or inconsistent extraction is
never accepted silently.


In [4]:
DNS_OVER_HTTPS_SERVER_NAME = "cloudflare-dns.com"
DNS_OVER_HTTPS_ADDRESSES = ["1.1.1.1", "1.0.0.1"]
DNS_OVER_HTTPS_TIMEOUT_SECONDS = 30
RESOLVED_HOST_ADDRESSES: dict[str, list[str]] = {}


class FixedAddressHTTPSConnection(HTTPSConnection):
    """Open a TLS-verified HTTPS connection without local DNS lookup."""

    def __init__(self, host: str, fixed_address: str, timeout: float) -> None:
        super().__init__(host, timeout=timeout)
        self.fixed_address = fixed_address

    def connect(self) -> None:
        """Connect to a fixed address while validating the host certificate."""
        self.sock = socket.create_connection(
            (self.fixed_address, self.port),
            self.timeout,
            self.source_address,
        )
        if self._tunnel_host:
            self._tunnel()
        self.sock = self._context.wrap_socket(
            self.sock,
            server_hostname=self.host,
        )


def find_dns_error(error: BaseException) -> socket.gaierror | None:
    """Return a nested DNS-resolution error when one caused the failure."""
    current_error: BaseException | None = error
    visited_error_ids: set[int] = set()
    while current_error is not None and id(current_error) not in visited_error_ids:
        visited_error_ids.add(id(current_error))
        if isinstance(current_error, socket.gaierror):
            return current_error
        if isinstance(current_error, URLError) and isinstance(
            current_error.reason,
            BaseException,
        ):
            current_error = current_error.reason
            continue
        current_error = current_error.__cause__ or current_error.__context__
    return None


def request_headers() -> dict[str, str]:
    """Return API headers without exposing the optional application token."""
    headers = {
        "Accept": "application/json",
        "User-Agent": "urban-operations-intelligence-day4/1.0",
    }
    if APP_TOKEN:
        headers["X-App-Token"] = APP_TOKEN
    return headers


def resolve_ipv4_with_dns_over_https(host: str) -> list[str]:
    """Resolve public IPv4 addresses through TLS-verified DNS-over-HTTPS."""
    if host in RESOLVED_HOST_ADDRESSES:
        return RESOLVED_HOST_ADDRESSES[host]
    request_path = "/dns-query?" + urlencode({"name": host, "type": "A"})
    last_error: BaseException | None = None
    for resolver_address in DNS_OVER_HTTPS_ADDRESSES:
        connection = FixedAddressHTTPSConnection(
            DNS_OVER_HTTPS_SERVER_NAME,
            resolver_address,
            timeout=DNS_OVER_HTTPS_TIMEOUT_SECONDS,
        )
        try:
            connection.request(
                "GET",
                request_path,
                headers={
                    "Accept": "application/dns-json",
                    "User-Agent": "urban-operations-intelligence-day4/1.0",
                },
            )
            response = connection.getresponse()
            response_body = response.read()
            if response.status != 200:
                raise ConnectionError(
                    f"DNS-over-HTTPS returned HTTP {response.status}."
                )
            payload = json.loads(response_body)
            addresses = [
                str(answer["data"])
                for answer in payload.get("Answer", [])
                if answer.get("type") == 1
                and isinstance(answer.get("data"), str)
            ]
            for address in addresses:
                socket.inet_aton(address)
            if addresses:
                RESOLVED_HOST_ADDRESSES[host] = list(dict.fromkeys(addresses))
                return RESOLVED_HOST_ADDRESSES[host]
            raise ConnectionError(f"No IPv4 addresses returned for {host}.")
        except (
            OSError,
            TimeoutError,
            ConnectionError,
            json.JSONDecodeError,
        ) as error:
            last_error = error
        finally:
            connection.close()
    raise ConnectionError(
        f"Unable to resolve {host} through the encrypted DNS fallback."
    ) from last_error


def fetch_json_from_resolved_addresses(
    url: str,
    host: str,
    addresses: list[str],
    timeout_seconds: float,
) -> object:
    """Fetch JSON from fixed addresses while retaining TLS hostname checks."""
    parsed_url = urlsplit(url)
    request_target = parsed_url.path
    if parsed_url.query:
        request_target += f"?{parsed_url.query}"
    last_error: BaseException | None = None
    for address in addresses:
        connection = FixedAddressHTTPSConnection(
            host,
            address,
            timeout=timeout_seconds,
        )
        try:
            connection.request("GET", request_target, headers=request_headers())
            response = connection.getresponse()
            response_body = response.read()
            if response.status >= 400:
                error_text = response_body.decode("utf-8", errors="replace")
                raise RuntimeError(
                    f"HTTP {response.status} {response.reason}: "
                    f"{error_text[:1000]}"
                )
            return json.loads(response_body)
        except (
            OSError,
            TimeoutError,
            ConnectionError,
            RuntimeError,
            json.JSONDecodeError,
        ) as error:
            last_error = error
        finally:
            connection.close()
    raise ConnectionError(f"Unable to reach {host} by resolved address.") from last_error


def fetch_json_url(
    url: str,
    *,
    timeout_seconds: float = API_TIMEOUT_SECONDS,
    max_attempts: int = API_MAX_ATTEMPTS,
) -> object:
    """Return JSON using bounded retries and a secure cached DNS fallback."""
    host = urlsplit(url).hostname
    if not host:
        raise ValueError(f"URL does not contain a host: {url}")
    if host in RESOLVED_HOST_ADDRESSES:
        return fetch_json_from_resolved_addresses(
            url,
            host,
            RESOLVED_HOST_ADDRESSES[host],
            timeout_seconds,
        )

    last_error: BaseException | None = None
    for attempt in range(1, max_attempts + 1):
        request = Request(url, headers=request_headers())
        try:
            with urlopen(request, timeout=timeout_seconds) as response:
                return json.load(response)
        except HTTPError as error:
            error_body = error.read().decode("utf-8", errors="replace")
            http_error = RuntimeError(
                f"HTTP {error.code} {error.reason}: {error_body[:2000]}"
            )
            if error.code not in {408, 429, 500, 502, 503, 504}:
                raise http_error from error
            last_error = http_error
        except (
            URLError,
            TimeoutError,
            socket.timeout,
            RemoteDisconnected,
            json.JSONDecodeError,
        ) as error:
            last_error = error
            if find_dns_error(error) is not None:
                addresses = resolve_ipv4_with_dns_over_https(host)
                return fetch_json_from_resolved_addresses(
                    url,
                    host,
                    addresses,
                    timeout_seconds,
                )
        if attempt < max_attempts:
            time.sleep(API_RETRY_BACKOFF_SECONDS ** (attempt - 1))
    raise RuntimeError(
        f"Read-only request failed after {max_attempts} attempts."
    ) from last_error


def execute_soql(query: str) -> list[dict[str, object]]:
    """Execute one SoQL query and validate the JSON record structure."""
    query_url = f"{API_ENDPOINT}?{urlencode({'$query': query})}"
    payload = fetch_json_url(query_url)
    if not isinstance(payload, list):
        raise RuntimeError("NYC Open Data returned JSON that was not a list.")
    records = [record for record in payload if isinstance(record, dict)]
    if len(records) != len(payload):
        raise RuntimeError("NYC Open Data returned a non-object row.")
    return records


def quote_soql(value: str) -> str:
    """Return one safely escaped SoQL string literal."""
    return "'" + value.replace("'", "''") + "'"


def build_scope_condition(snapshot_end_date: str | None = None) -> str:
    """Build the configured DOT and created-date filter."""
    conditions = [
        f"agency = {quote_soql(AGENCY)}",
        f"created_date >= {quote_soql(CONFIGURED_START_DATE)}",
    ]
    configured_upper_bound = CONFIGURED_END_DATE or snapshot_end_date
    if configured_upper_bound:
        conditions.append(
            f"created_date <= {quote_soql(configured_upper_bound)}"
        )
    return " AND ".join(conditions)


def fetch_scope_record(scope_condition: str) -> dict[str, object]:
    """Return authoritative count and creation-date bounds for a scope."""
    records = execute_soql(
        "SELECT count(*) AS row_count, "
        "min(created_date) AS minimum_created_date, "
        "max(created_date) AS maximum_created_date "
        f"WHERE {scope_condition}"
    )
    if len(records) != 1:
        raise RuntimeError(
            f"Scope query returned {len(records)} records instead of one."
        )
    return records[0]


def fetch_full_dot_extract(
    snapshot_condition: str,
    expected_row_count: int,
) -> tuple[pd.DataFrame, int]:
    """Retrieve and validate every configured DOT record by pagination."""
    selected_columns = ", ".join(EXTRACTION_COLUMNS)
    page_frames: list[pd.DataFrame] = []
    offset = 0
    page_number = 1

    while offset < expected_row_count:
        page_query = (
            f"SELECT {selected_columns} "
            f"WHERE {snapshot_condition} "
            f"ORDER BY {API_ORDERING} "
            f"LIMIT {API_PAGE_SIZE} OFFSET {offset}"
        )
        records = execute_soql(page_query)
        if not records:
            raise RuntimeError(
                "Pagination ended before the expected row count was reached: "
                f"retrieved {offset:,} of {expected_row_count:,} rows."
            )
        page_frame = pd.DataFrame.from_records(records).reindex(
            columns=EXTRACTION_COLUMNS
        )
        page_frames.append(page_frame)
        offset += len(page_frame)
        print(
            f"Loaded page {page_number:,}: {len(page_frame):,} rows "
            f"({offset:,}/{expected_row_count:,})."
        )
        if len(page_frame) < API_PAGE_SIZE and offset < expected_row_count:
            raise RuntimeError(
                "A short API page was returned before extraction completed."
            )
        page_number += 1

    frame = pd.concat(page_frames, ignore_index=True)
    if len(frame) != expected_row_count:
        raise RuntimeError(
            "Pagination row-count mismatch: "
            f"expected {expected_row_count:,}, retrieved {len(frame):,}."
        )
    if frame["unique_key"].isna().any():
        raise RuntimeError("The paginated extract contains null unique_key values.")
    duplicated_identifier_count = int(frame["unique_key"].duplicated().sum())
    if duplicated_identifier_count:
        raise RuntimeError(
            "Deterministic pagination produced duplicate unique_key values: "
            f"{duplicated_identifier_count:,} duplicates."
        )
    return frame, page_number - 1


In [5]:
extraction_started_at_utc = datetime.now(timezone.utc)
metadata_payload = fetch_json_url(DATASET_METADATA_URL)
if not isinstance(metadata_payload, dict):
    raise RuntimeError("Dataset metadata response was not a JSON object.")
metadata_columns = metadata_payload.get("columns")
if not isinstance(metadata_columns, list):
    raise RuntimeError("Dataset metadata did not include a column list.")
source_columns = {
    str(column["fieldName"])
    for column in metadata_columns
    if isinstance(column, dict) and column.get("fieldName")
}

initial_scope_record = fetch_scope_record(build_scope_condition())
snapshot_maximum_created_date = initial_scope_record.get("maximum_created_date")
if not snapshot_maximum_created_date:
    raise ValueError(
        "The configured DOT API query returned no records. "
        "Target feasibility cannot be evaluated."
    )
snapshot_condition = build_scope_condition(str(snapshot_maximum_created_date))
snapshot_scope_record = fetch_scope_record(snapshot_condition)
expected_row_count = int(snapshot_scope_record["row_count"])
if expected_row_count == 0:
    raise ValueError(
        "The configured DOT API query returned no records. "
        "Target feasibility cannot be evaluated."
    )

df, api_page_count = fetch_full_dot_extract(
    snapshot_condition,
    expected_row_count,
)
extraction_completed_at_utc = datetime.now(timezone.utc)
final_scope_record = fetch_scope_record(snapshot_condition)
final_source_row_count = int(final_scope_record["row_count"])
if final_source_row_count != expected_row_count:
    raise RuntimeError(
        "The live API scope changed during pagination: "
        f"started with {expected_row_count:,} rows and ended with "
        f"{final_source_row_count:,}. Rerun all cells for a consistent snapshot."
    )
if df.empty:
    raise ValueError(
        "The configured DOT API query returned no records. "
        "Target feasibility cannot be evaluated."
    )
assert len(df) > 0

len(df), api_page_count, snapshot_maximum_created_date


Loaded page 1: 50,000 rows (50,000/1,469,836).


Loaded page 2: 50,000 rows (100,000/1,469,836).


Loaded page 3: 50,000 rows (150,000/1,469,836).


Loaded page 4: 50,000 rows (200,000/1,469,836).


Loaded page 5: 50,000 rows (250,000/1,469,836).


...

Loaded page 27: 50,000 rows (1,350,000/1,469,836).


Loaded page 28: 50,000 rows (1,400,000/1,469,836).


Loaded page 29: 50,000 rows (1,450,000/1,469,836).


Loaded page 30: 19,836 rows (1,469,836/1,469,836).


(1469836, 30, '2026-07-24T01:31:13.000')

## 8. Extraction and scope verification

The summary below is calculated from the current run. The maximum creation
timestamp is the fixed live-snapshot boundary used on every page.


In [6]:
raw_created_dates_for_metadata = pd.to_datetime(
    df["created_date"],
    errors="coerce",
    utc=True,
)
observed_agencies = sorted(
    df["agency"].dropna().astype("string").unique().tolist()
)
if observed_agencies != [AGENCY]:
    raise ValueError(
        "Expected a DOT-only extract but found agencies: "
        f"{observed_agencies}"
    )

extraction_summary = pd.DataFrame(
    [
        {
            "api_endpoint": API_ENDPOINT,
            "agency_filter": AGENCY,
            "configured_start_date": CONFIGURED_START_DATE,
            "configured_end_date": CONFIGURED_END_DATE or "live snapshot",
            "snapshot_maximum_created_date": snapshot_maximum_created_date,
            "extraction_started_at_utc": extraction_started_at_utc.isoformat(),
            "extraction_completed_at_utc": extraction_completed_at_utc.isoformat(),
            "api_pages": api_page_count,
            "rows_retrieved": len(df),
            "columns_retrieved": len(df.columns),
            "minimum_created_date": raw_created_dates_for_metadata.min(),
            "maximum_created_date": raw_created_dates_for_metadata.max(),
        }
    ]
)
agency_scope_summary = (
    df.groupby("agency", dropna=False)
    .size()
    .rename("record_count")
    .reset_index()
)
display(extraction_summary.T)
display(agency_scope_summary)


,0
api_endpoint,https://data.cityofnewyork.us/resource/erm2-nwe9.json
agency_filter,DOT
configured_start_date,2020-01-01T00:00:00.000
configured_end_date,live snapshot
snapshot_maximum_created_date,2026-07-24T01:31:13.000
extraction_started_at_utc,2026-07-25T06:04:22.121138+00:00
extraction_completed_at_utc,2026-07-25T06:19:07.097895+00:00
api_pages,30
rows_retrieved,1469836
columns_retrieved,10


,agency,record_count
0,DOT,1469836


**Interpretation.** The extraction must contain DOT and only DOT. Alternative
agencies are deliberately outside this notebook. A scope mismatch raises an
error rather than allowing the analysis to continue.


## 9. Required-column validation


In [7]:
required_column_check = pd.DataFrame(
    [
        {
            "column": column,
            "present": column in source_columns,
            "purpose": TARGET_COLUMN_PURPOSES[column],
        }
        for column in TARGET_REQUIRED_COLUMNS
    ]
)
missing_required_columns = [
    column
    for column in TARGET_REQUIRED_COLUMNS
    if column not in source_columns
]
display(required_column_check)
if missing_required_columns:
    raise KeyError(
        "Missing target-required columns: "
        f"{missing_required_columns}"
    )
assert set(TARGET_REQUIRED_COLUMNS).issubset(source_columns)
assert set(TARGET_REQUIRED_COLUMNS).issubset(df.columns)


,column,present,purpose
0,unique_key,True,Complaint identifier
1,created_date,True,Creation timestamp and temporal context
2,closed_date,True,Actual complaint closure timestamp
3,due_date,True,Expected resolution deadline
4,status,True,Supports eligibility and lifecycle interpretation


**Interpretation.** `present` is validated against authoritative source
metadata, not inferred from returned JSON keys. This distinguishes a missing
source column from a column such as `due_date` that exists in the schema but
may be null for every selected record.


## 10. Raw timestamp preservation


In [8]:
raw_target_dates = df[TARGET_DATE_COLUMNS].copy()


Raw target timestamps are retained separately so source missingness can be
distinguished from conversion failures. No timestamp is imputed or replaced.


## 11. Timestamp parsing


In [9]:
for column in TARGET_DATE_COLUMNS:
    df[column] = pd.to_datetime(
        df[column],
        errors="coerce",
        utc=True,
    )


## 12. Timestamp parse-quality summary


In [10]:
timestamp_parse_rows = []
for column in TARGET_DATE_COLUMNS:
    raw_present = raw_target_dates[column].notna()
    parsed_missing = df[column].isna()
    raw_non_null_count = int(raw_present.sum())
    parse_failure_count = int((raw_present & parsed_missing).sum())
    timestamp_parse_rows.append(
        {
            "column": column,
            "total_rows": len(df),
            "raw_non_null_count": raw_non_null_count,
            "raw_missing_count": int((~raw_present).sum()),
            "parsed_non_null_count": int(df[column].notna().sum()),
            "parsed_missing_count": int(parsed_missing.sum()),
            "parse_failure_count": parse_failure_count,
            "parse_failure_pct_of_raw_non_null": (
                100.0 * parse_failure_count / raw_non_null_count
                if raw_non_null_count
                else 0.0
            ),
        }
    )
timestamp_parse_summary = pd.DataFrame(timestamp_parse_rows)
display(timestamp_parse_summary.round(6))


,column,total_rows,raw_non_null_count,raw_missing_count,parsed_non_null_count,parsed_missing_count,parse_failure_count,parse_failure_pct_of_raw_non_null
0,created_date,1469836,1469836,0,1469836,0,0,0.0
1,closed_date,1469836,1437880,31956,1437880,31956,0,0.0
2,due_date,1469836,0,1469836,0,1469836,0,0.0


**Interpretation.** Source nulls are not parsing failures. In particular,
zero `due_date` parsing failures can coexist with zero availability because
there were no source values to parse. Parse quality and field coverage answer
different questions.


## 13. Target-field coverage


In [11]:
def build_coverage_summary(
    frame: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    """Return non-null and missing coverage for selected columns."""
    total_rows = len(frame)
    rows = []
    for column in columns:
        non_null_count = int(frame[column].notna().sum())
        missing_count = total_rows - non_null_count
        rows.append(
            {
                "column": column,
                "total_rows": total_rows,
                "non_null_count": non_null_count,
                "missing_count": missing_count,
                "coverage_pct": (
                    100.0 * non_null_count / total_rows if total_rows else 0.0
                ),
                "missing_pct": (
                    100.0 * missing_count / total_rows if total_rows else 0.0
                ),
            }
        )
    return pd.DataFrame(rows)


target_field_coverage = build_coverage_summary(
    df,
    TARGET_REQUIRED_COLUMNS,
)
display(target_field_coverage.round(6))


,column,total_rows,non_null_count,missing_count,coverage_pct,missing_pct
0,unique_key,1469836,1469836,0,100.00000,0.00000
1,created_date,1469836,1469836,0,100.00000,0.00000
2,closed_date,1469836,1437880,31956,97.82588,2.17412
3,due_date,1469836,0,1469836,0.00000,100.00000
4,status,1469836,1469836,0,100.00000,0.00000


**Interpretation.** Coverage is calculated from the full paginated extract.
The target cannot be evaluated for a row unless the relevant timestamp
columns are both populated, regardless of whether the columns exist in the
source schema.


## 14. Explicit due-date blocker assessment


In [12]:
due_date_non_null_count = int(df["due_date"].notna().sum())
due_date_available = due_date_non_null_count > 0
due_date_coverage_pct = (
    100.0 * due_date_non_null_count / len(df) if len(df) else 0.0
)
due_date_blocker_assessment = pd.DataFrame(
    [
        {"metric": "Due Date available", "value": due_date_available},
        {"metric": "Usable Due Date records", "value": due_date_non_null_count},
        {"metric": "Due Date coverage percentage", "value": due_date_coverage_pct},
    ]
)
display(due_date_blocker_assessment)


,metric,value
0,Due Date available,False
1,Usable Due Date records,0
2,Due Date coverage percentage,0.0


The zero/non-zero decision above is calculated rather than assumed. The
notebook continues even if no due dates exist so every subgroup and
non-evaluable timestamp comparison is documented.


## 15. Target-eligibility definition


In [13]:
has_created_date = df["created_date"].notna()
has_closed_date = df["closed_date"].notna()
has_due_date = df["due_date"].notna()
has_closed_and_due_date = has_closed_date & has_due_date
target_eligible = (
    has_created_date
    & has_closed_date
    & has_due_date
)
df["target_eligible"] = target_eligible


`target_eligible` is only an eligibility indicator. It is not the prediction
label and does not classify an ineligible complaint as on time.


## 16. Eligibility summary


In [14]:
def percentage_of_total(count: int, total: int) -> float:
    """Return a safe percentage using the full extract denominator."""
    return 100.0 * count / total if total else 0.0


eligibility_counts = {
    "total_records": len(df),
    "has_created_date": int(has_created_date.sum()),
    "has_closed_date": int(has_closed_date.sum()),
    "has_due_date": int(has_due_date.sum()),
    "has_closed_and_due_date": int(has_closed_and_due_date.sum()),
    "fully_target_eligible": int(df["target_eligible"].sum()),
}
eligibility_summary = pd.DataFrame(
    [
        {
            "metric": metric,
            "count": count,
            "percentage_of_total": percentage_of_total(count, len(df)),
        }
        for metric, count in eligibility_counts.items()
    ]
)
assert (
    eligibility_summary.loc[
        eligibility_summary["metric"] == "fully_target_eligible",
        "count",
    ].iloc[0]
    == int(df["target_eligible"].sum())
)
display(eligibility_summary.round(6))


,metric,count,percentage_of_total
0,total_records,1469836,100.00000
1,has_created_date,1469836,100.00000
2,has_closed_date,1437880,97.82588
3,has_due_date,0,0.00000
4,has_closed_and_due_date,0,0.00000
5,fully_target_eligible,0,0.00000


**Interpretation.** Full target eligibility requires usable creation,
closure, and due timestamps on the same record. Partial timestamp coverage
is not sufficient to construct the label.


## 17. Ineligibility-reason analysis


In [15]:
def assign_target_eligibility_reason(frame: pd.DataFrame) -> pd.Series:
    """Assign one mutually exclusive target-eligibility reason per row."""
    reason = pd.Series("eligible", index=frame.index, dtype="string")
    missing_created = frame["created_date"].isna()
    missing_closed = frame["closed_date"].isna()
    missing_due = frame["due_date"].isna()
    reason.loc[
        ~missing_created & ~missing_closed & missing_due
    ] = "missing_due_date_only"
    reason.loc[
        ~missing_created & missing_closed & ~missing_due
    ] = "missing_closed_date_only"
    reason.loc[
        ~missing_created & missing_closed & missing_due
    ] = "missing_closed_and_due_date"
    reason.loc[missing_created] = "missing_created_date"
    return reason


df["target_eligibility_reason"] = assign_target_eligibility_reason(df)
ineligibility_reason_summary = (
    df["target_eligibility_reason"]
    .value_counts(dropna=False)
    .rename_axis("reason")
    .rename("record_count")
    .reset_index()
)
ineligibility_reason_summary["percentage_of_total"] = (
    ineligibility_reason_summary["record_count"] / len(df) * 100.0
)
assert len(df["target_eligibility_reason"]) == len(df)
assert ineligibility_reason_summary["record_count"].sum() == len(df)
dominant_ineligibility_reason = str(
    ineligibility_reason_summary.iloc[0]["reason"]
)
display(ineligibility_reason_summary.round(6))


,reason,record_count,percentage_of_total
0,missing_due_date_only,1437880,97.82588
1,missing_closed_and_due_date,31956,2.17412


**Interpretation.** Reasons are mutually exclusive and reconcile to the full
row count. The leading row identifies the dominant blocker without double
counting records that lack multiple timestamps.


## 18. Grouped target-coverage helper


In [16]:
def build_grouped_target_coverage(
    frame: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:
    """Return complete timestamp and eligibility coverage by one group."""
    grouped = (
        frame.groupby(group_column, dropna=False)
        .agg(
            record_count=("unique_key", "size"),
            created_date_count=("created_date", "count"),
            closed_date_count=("closed_date", "count"),
            due_date_count=("due_date", "count"),
            target_eligible_count=("target_eligible", "sum"),
        )
        .reset_index()
    )
    grouped["target_eligible_count"] = grouped[
        "target_eligible_count"
    ].astype("int64")
    for count_column, percentage_column in [
        ("created_date_count", "created_date_coverage_pct"),
        ("closed_date_count", "closed_date_coverage_pct"),
        ("due_date_count", "due_date_coverage_pct"),
        ("target_eligible_count", "target_eligible_pct"),
    ]:
        grouped[percentage_column] = (
            grouped[count_column] / grouped["record_count"] * 100.0
        )
    return grouped


## 19. Coverage by complaint type


In [17]:
coverage_by_complaint_type = build_grouped_target_coverage(
    df,
    "complaint_type",
).sort_values(
    ["record_count", "complaint_type"],
    ascending=[False, True],
    na_position="last",
)
complaint_types_with_due_dates = coverage_by_complaint_type.loc[
    coverage_by_complaint_type["due_date_count"] > 0
]
complaint_types_with_eligible_records = coverage_by_complaint_type.loc[
    coverage_by_complaint_type["target_eligible_count"] > 0
]
complaint_type_coverage_summary = pd.DataFrame(
    [
        {
            "number_of_complaint_types": len(coverage_by_complaint_type),
            "complaint_types_with_due_dates": len(
                complaint_types_with_due_dates
            ),
            "complaint_types_with_eligible_records": len(
                complaint_types_with_eligible_records
            ),
            "maximum_due_date_coverage_pct": (
                float(coverage_by_complaint_type["due_date_coverage_pct"].max())
                if not coverage_by_complaint_type.empty
                else 0.0
            ),
        }
    ]
)
display(complaint_type_coverage_summary.round(6))
display(coverage_by_complaint_type.head(20).round(6))


,number_of_complaint_types,complaint_types_with_due_dates,complaint_types_with_eligible_records,maximum_due_date_coverage_pct
0,35,0,0,0.0


,complaint_type,record_count,created_date_count,closed_date_count,due_date_count,target_eligible_count,created_date_coverage_pct,closed_date_coverage_pct,due_date_coverage_pct,target_eligible_pct
27,Street Condition,497097,497097,477066,0,0,100.0,95.970404,0.0,0.0
28,Street Light Condition,297055,297055,289055,0,0,100.0,97.306896,0.0,0.0
32,Traffic Signal Condition,275602,275602,275366,0,0,100.0,99.914369,0.0,0.0
26,Sidewalk Condition,160942,160942,159477,0,0,100.0,99.089734,0.0,0.0
8,Curb Condition,43880,43880,43194,0,0,100.0,98.436645,0.0,0.0
4,Broken Parking Meter,34468,34468,34384,0,0,100.0,99.756296,0.0,0.0
29,Street Sign - Damaged,32962,32962,32583,0,0,100.0,98.850191,0.0,0.0
23,Outdoor Dining,32011,32011,31782,0,0,100.0,99.284621,0.0,0.0
31,Street Sign - Missing,25918,25918,25522,0,0,100.0,98.472104,0.0,0.0
17,Highway Condition,23863,23863,23734,0,0,100.0,99.459414,0.0,0.0


In [18]:
display(
    Markdown(
        "**Interpretation.** "
        + (
            "No complaint type contains a due date or an eligible record; "
            "the conclusion uses the complete grouped table, including small "
            "categories not shown in the preview."
            if complaint_types_with_due_dates.empty
            and complaint_types_with_eligible_records.empty
            else (
                f"{len(complaint_types_with_due_dates):,} complaint type(s) "
                "contain at least one due date and require further review; "
                f"{len(complaint_types_with_eligible_records):,} contain an "
                "eligible record."
            )
        )
    )
)


**Interpretation.** No complaint type contains a due date or an eligible record; the conclusion uses the complete grouped table, including small categories not shown in the preview.

## 20. Coverage by status


In [19]:
coverage_by_status = build_grouped_target_coverage(
    df,
    "status",
).sort_values(
    ["record_count", "status"],
    ascending=[False, True],
    na_position="last",
)
statuses_with_due_dates = coverage_by_status.loc[
    coverage_by_status["due_date_count"] > 0
]
statuses_with_eligible_records = coverage_by_status.loc[
    coverage_by_status["target_eligible_count"] > 0
]
status_coverage_summary = pd.DataFrame(
    [
        {
            "number_of_statuses": len(coverage_by_status),
            "statuses_with_due_dates": len(statuses_with_due_dates),
            "statuses_with_eligible_records": len(
                statuses_with_eligible_records
            ),
        }
    ]
)
display(status_coverage_summary)
display(coverage_by_status.round(6))


,number_of_statuses,statuses_with_due_dates,statuses_with_eligible_records
0,7,0,0


,status,record_count,created_date_count,closed_date_count,due_date_count,target_eligible_count,created_date_coverage_pct,closed_date_coverage_pct,due_date_coverage_pct,target_eligible_pct
2,Closed,1387691,1387691,1387583,0,0,100.0,99.992217,0.0,0.0
5,Pending,61534,61534,45167,0,0,100.0,73.401697,0.0,0.0
0,Assigned,10975,10975,5130,0,0,100.0,46.742597,0.0,0.0
3,In Progress,5058,5058,0,0,0,100.0,0.000000,0.0,0.0
4,Open,4560,4560,0,0,0,100.0,0.000000,0.0,0.0
6,Unspecified,17,17,0,0,0,100.0,0.000000,0.0,0.0
1,Cancel,1,1,0,0,0,100.0,0.000000,0.0,0.0


**Interpretation.** A complaint marked `Closed` has an actual outcome state,
but without an expected `due_date` it cannot be classified as on time or
late. Open complaints without closure timestamps are historically
unlabelled. Cancellation- or duplicate-like statuses remain visible here,
but they require a documented eligibility rule after a viable dataset is
selected; no status-only relabelling is performed.


## 21. Coverage by created year


In [20]:
df["created_year"] = df["created_date"].dt.year.astype("Int64")
coverage_by_year = build_grouped_target_coverage(
    df,
    "created_year",
).sort_values("created_year", na_position="last")
years_with_due_dates = coverage_by_year.loc[
    coverage_by_year["due_date_count"] > 0
]
years_with_eligible_records = coverage_by_year.loc[
    coverage_by_year["target_eligible_count"] > 0
]
year_coverage_summary = pd.DataFrame(
    [
        {
            "years_represented": len(coverage_by_year),
            "years_with_non_zero_due_date_coverage": len(
                years_with_due_dates
            ),
            "years_with_eligible_records": len(years_with_eligible_records),
        }
    ]
)
display(year_coverage_summary)
display(coverage_by_year.round(6))


,years_represented,years_with_non_zero_due_date_coverage,years_with_eligible_records
0,7,0,0


,created_year,record_count,created_date_count,closed_date_count,due_date_count,target_eligible_count,created_date_coverage_pct,closed_date_coverage_pct,due_date_coverage_pct,target_eligible_pct
0,2020,212778,212778,210699,0,0,100.0,99.022925,0.0,0.0
1,2021,226188,226188,224407,0,0,100.0,99.212602,0.0,0.0
2,2022,238204,238204,236414,0,0,100.0,99.248543,0.0,0.0
3,2023,193947,193947,192318,0,0,100.0,99.160080,0.0,0.0
4,2024,204807,204807,200886,0,0,100.0,98.085515,0.0,0.0
5,2025,212108,212108,206751,0,0,100.0,97.474400,0.0,0.0
6,2026,181804,181804,166405,0,0,100.0,91.529889,0.0,0.0


**Interpretation.** The complete year table determines whether due dates are
confined to a historical period. No year is assumed to behave like another.


## 22. DOT agency-scope confirmation


In [21]:
coverage_by_agency = build_grouped_target_coverage(
    df,
    "agency",
).sort_values("agency", na_position="last")
assert coverage_by_agency["agency"].astype("string").tolist() == [AGENCY]
display(coverage_by_agency.round(6))

if "agency_name" in df.columns:
    agency_name_summary = (
        df.groupby("agency_name", dropna=False)
        .size()
        .rename("record_count")
        .reset_index()
        .sort_values("record_count", ascending=False)
    )
    display(agency_name_summary)


,agency,record_count,created_date_count,closed_date_count,due_date_count,target_eligible_count,created_date_coverage_pct,closed_date_coverage_pct,due_date_coverage_pct,target_eligible_pct
0,DOT,1469836,1469836,1437880,0,0,100.0,97.82588,0.0,0.0


,agency_name,record_count
0,Department of Transportation,1469836


**Interpretation.** This is scope confirmation only: the current extract
contains DOT records only. No alternative agency is queried or compared.


## 23. Timestamp consistency checks


In [22]:
closed_created_evaluable = (
    df["closed_date"].notna() & df["created_date"].notna()
)
due_created_evaluable = (
    df["due_date"].notna() & df["created_date"].notna()
)
closed_due_evaluable = (
    df["closed_date"].notna() & df["due_date"].notna()
)

timestamp_check_definitions = [
    {
        "check_name": "closed_date_before_created_date",
        "required_columns": "created_date, closed_date",
        "evaluable_mask": closed_created_evaluable,
        "matching_mask": (
            closed_created_evaluable
            & (df["closed_date"] < df["created_date"])
        ),
        "interpretation": (
            "Chronology violation: closure precedes complaint creation."
        ),
    },
    {
        "check_name": "due_date_before_created_date",
        "required_columns": "created_date, due_date",
        "evaluable_mask": due_created_evaluable,
        "matching_mask": (
            due_created_evaluable
            & (df["due_date"] < df["created_date"])
        ),
        "interpretation": (
            "Deadline precedes complaint creation; review source semantics."
        ),
    },
    {
        "check_name": "closed_date_after_due_date",
        "required_columns": "closed_date, due_date",
        "evaluable_mask": closed_due_evaluable,
        "matching_mask": (
            closed_due_evaluable
            & (df["closed_date"] > df["due_date"])
        ),
        "interpretation": "Candidate positive target condition.",
    },
    {
        "check_name": "closed_date_equal_to_due_date",
        "required_columns": "closed_date, due_date",
        "evaluable_mask": closed_due_evaluable,
        "matching_mask": (
            closed_due_evaluable
            & (df["closed_date"] == df["due_date"])
        ),
        "interpretation": "Exact deadline closure; candidate class 0.",
    },
]

timestamp_check_rows = []
for check in timestamp_check_definitions:
    evaluable_row_count = int(check["evaluable_mask"].sum())
    matching_count = int(check["matching_mask"].sum())
    timestamp_check_rows.append(
        {
            "check_name": check["check_name"],
            "required_columns": check["required_columns"],
            "evaluable_row_count": evaluable_row_count,
            "matching_or_violation_count": matching_count,
            "percentage_of_evaluable_rows": (
                100.0 * matching_count / evaluable_row_count
                if evaluable_row_count
                else pd.NA
            ),
            "evaluation_status": (
                "evaluated" if evaluable_row_count else "not_evaluable"
            ),
            "interpretation": check["interpretation"],
        }
    )
timestamp_checks = pd.DataFrame(timestamp_check_rows)
display(timestamp_checks.round(6))


,check_name,required_columns,evaluable_row_count,matching_or_violation_count,percentage_of_evaluable_rows,evaluation_status,interpretation
0,closed_date_before_created_date,"created_date, closed_date",1437880,45253,3.147203,evaluated,Chronology violation: closure precedes complaint creation.
1,due_date_before_created_date,"created_date, due_date",0,0,<NA>,not_evaluable,Deadline precedes complaint creation; review source semantics.
2,closed_date_after_due_date,"closed_date, due_date",0,0,<NA>,not_evaluable,Candidate positive target condition.
3,closed_date_equal_to_due_date,"closed_date, due_date",0,0,<NA>,not_evaluable,Exact deadline closure; candidate class 0.


**Interpretation.** Each comparison has its own evaluable mask. A result of
zero matches among zero evaluable rows is `not_evaluable`, never a passed
check. Comparisons involving `due_date` cannot be interpreted when the
extract supplies no due dates.


## 24. Safe target-construction attempt


In [23]:
eligible_df = df.loc[df["target_eligible"]].copy()
if eligible_df.empty:
    target_creation_status = "not_created"
    target_creation_reason = (
        "No records contain all timestamps required to construct the target."
    )
else:
    eligible_df[TARGET_NAME] = (
        eligible_df["closed_date"] > eligible_df["due_date"]
    ).astype("int8")
    observed_target_values = set(
        eligible_df[TARGET_NAME].dropna().unique().tolist()
    )
    if not observed_target_values.issubset({0, 1}):
        raise ValueError(
            "Target contains unexpected values: "
            f"{observed_target_values}"
        )
    assert set(
        eligible_df[TARGET_NAME].dropna().unique()
    ).issubset({0, 1})
    target_creation_status = "created"
    target_creation_reason = (
        "Target created for eligible historical records."
    )


**Interpretation.** The comparison is applied only to records that contain
all required timestamps. Missing deadlines are never interpreted as an
on-time label, and no missing target is filled with zero.


## 25. Target result summary


In [24]:
eligible_record_count = len(eligible_df)
eligible_record_percentage = percentage_of_total(
    eligible_record_count,
    len(df),
)
if target_creation_status == "created":
    class_counts = (
        eligible_df[TARGET_NAME]
        .value_counts()
        .reindex([0, 1], fill_value=0)
    )
    target_class_summary = pd.DataFrame(
        [
            {
                "target_class": target_class,
                "record_count": int(class_counts.loc[target_class]),
                "percentage_of_eligible": percentage_of_total(
                    int(class_counts.loc[target_class]),
                    eligible_record_count,
                ),
            }
            for target_class in [0, 1]
        ]
    )
    observed_target_classes = ", ".join(
        map(
            str,
            sorted(
                eligible_df[TARGET_NAME]
                .dropna()
                .unique()
                .tolist()
            ),
        )
    )
    display(target_class_summary.round(6))
else:
    observed_target_classes = "not_created"

target_result_summary = pd.DataFrame(
    [
        {
            "target_name": TARGET_NAME,
            "target_creation_status": target_creation_status,
            "target_creation_reason": target_creation_reason,
            "eligible_record_count": eligible_record_count,
            "eligible_record_percentage": eligible_record_percentage,
            "observed_target_classes": observed_target_classes,
        }
    ]
)
display(target_result_summary)


,target_name,target_creation_status,target_creation_reason,eligible_record_count,eligible_record_percentage,observed_target_classes
0,missed_resolution_target,not_created,No records contain all timestamps required to construct the target.,0,0.0,not_created


## 26. Invalid alternative-target assessment


In [25]:
alternative_target_assessment = pd.DataFrame(
    [
        {
            "candidate": 'status == "Closed"',
            "valid_replacement": "No",
            "reason": (
                "Measures completion, not whether an expected deadline was missed."
            ),
        },
        {
            "candidate": "closed_date.isna()",
            "valid_replacement": "No",
            "reason": (
                "Mixes unresolved complaints with recently created complaints."
            ),
        },
        {
            "candidate": "Resolution duration above an arbitrary threshold",
            "valid_replacement": "No",
            "reason": "Invents a deadline not supplied by the source.",
        },
        {
            "candidate": "resolution_action_updated_date",
            "valid_replacement": "No",
            "reason": "Is not necessarily the expected resolution deadline.",
        },
        {
            "candidate": "resolution_description",
            "valid_replacement": "No",
            "reason": (
                "Is usually generated after operational action and risks leakage."
            ),
        },
        {
            "candidate": "Missing closed date",
            "valid_replacement": "No",
            "reason": (
                "Does not establish lateness without an expected deadline."
            ),
        },
    ]
)
display(alternative_target_assessment)


,candidate,valid_replacement,reason
0,"status == ""Closed""",No,"Measures completion, not whether an expected deadline was missed."
1,closed_date.isna(),No,Mixes unresolved complaints with recently created complaints.
2,Resolution duration above an arbitrary threshold,No,Invents a deadline not supplied by the source.
3,resolution_action_updated_date,No,Is not necessarily the expected resolution deadline.
4,resolution_description,No,Is usually generated after operational action and risks leakage.
5,Missing closed date,No,Does not establish lateness without an expected deadline.


Inventing any of these replacements would silently change the business
outcome. Open complaints cannot receive a historical closed-on-time versus
closed-late label under the candidate definition. Cancelled and duplicate
complaints need a documented eligibility rule after a source with usable
deadlines is selected; final status alone is not a labelling rule.


## 27. Prediction-time leakage assessment


In [26]:
leakage_assessment = pd.DataFrame(
    [
        {
            "field": "unique_key",
            "present_in_extract": "Yes",
            "target_construction_use": "Identifier only",
            "model_feature_use": "No",
            "decision_reason": "Record identifier, not predictive input.",
        },
        {
            "field": "created_date",
            "present_in_extract": "Yes",
            "target_construction_use": "Context and eligibility",
            "model_feature_use": "Yes, transformed",
            "decision_reason": "Available at complaint creation.",
        },
        {
            "field": "due_date",
            "present_in_extract": "Yes",
            "target_construction_use": "Required target boundary",
            "model_feature_use": "Conditional",
            "decision_reason": (
                "Must be verified as available at prediction time."
            ),
        },
        {
            "field": "closed_date",
            "present_in_extract": "Yes",
            "target_construction_use": "Required label input",
            "model_feature_use": "No",
            "decision_reason": "Known after the complaint is resolved.",
        },
        {
            "field": "status",
            "present_in_extract": "Yes",
            "target_construction_use": (
                "Supporting eligibility analysis"
            ),
            "model_feature_use": "Conditional",
            "decision_reason": (
                "Final status contains future outcome information."
            ),
        },
        {
            "field": "resolution_description",
            "present_in_extract": (
                "Yes" if "resolution_description" in source_columns else "No"
            ),
            "target_construction_use": "No",
            "model_feature_use": "No",
            "decision_reason": (
                "Generally post-resolution information."
            ),
        },
        {
            "field": "resolution_action_updated_date",
            "present_in_extract": (
                "Yes"
                if "resolution_action_updated_date" in source_columns
                else "No"
            ),
            "target_construction_use": "No",
            "model_feature_use": "No",
            "decision_reason": "Post-creation operational update.",
        },
    ]
)
display(leakage_assessment)


,field,present_in_extract,target_construction_use,model_feature_use,decision_reason
0,unique_key,Yes,Identifier only,No,"Record identifier, not predictive input."
1,created_date,Yes,Context and eligibility,"Yes, transformed",Available at complaint creation.
2,due_date,Yes,Required target boundary,Conditional,Must be verified as available at prediction time.
3,closed_date,Yes,Required label input,No,Known after the complaint is resolved.
4,status,Yes,Supporting eligibility analysis,Conditional,Final status contains future outcome information.
5,resolution_description,Yes,No,No,Generally post-resolution information.
6,resolution_action_updated_date,Yes,No,No,Post-creation operational update.


**Interpretation.** A field can be valid for label construction while still
being prohibited as a model feature. `closed_date`, final `status`, and
post-resolution fields are unavailable at the prediction moment and would
leak future information. This notebook documents roles; it does not perform
feature selection.


## 28. Programmatic feasibility decision


In [27]:
target_feasible = eligible_record_count > 0
if eligible_record_count == 0:
    feasibility_status = "NOT_FEASIBLE"
    feasibility_reason = (
        "No records in the current DOT extract contain all timestamps "
        f"required to construct {TARGET_NAME}."
    )
else:
    feasibility_status = "REQUIRES_FURTHER_REVIEW"
    feasibility_reason = (
        "Eligible records exist, but class balance, status exclusions, "
        "due-date semantics, and prediction-time availability require review."
    )

closed_date_non_null_count = int(df["closed_date"].notna().sum())
closed_date_coverage_pct = percentage_of_total(
    closed_date_non_null_count,
    len(df),
)
if due_date_non_null_count == 0:
    primary_blocker = "No usable due_date values"
elif closed_date_non_null_count == 0:
    primary_blocker = "No usable closed_date values"
elif eligible_record_count == 0:
    primary_blocker = "Required timestamps do not coexist on any record"
else:
    primary_blocker = "Further semantic and population review required"
recommended_next_step = (
    "Compare target-field coverage across alternative agencies and "
    "complaint types"
)

feasibility_decision = pd.DataFrame(
    [
        {
            "dataset_scope": "Current DOT API extract",
            "agency": AGENCY,
            "target_name": TARGET_NAME,
            "required_target_timestamps": (
                "created_date, closed_date, due_date"
            ),
            "total_record_count": len(df),
            "due_date_non_null_count": due_date_non_null_count,
            "due_date_coverage_pct": due_date_coverage_pct,
            "closed_date_non_null_count": closed_date_non_null_count,
            "closed_date_coverage_pct": closed_date_coverage_pct,
            "eligible_record_count": eligible_record_count,
            "eligible_record_pct": eligible_record_percentage,
            "target_creation_status": target_creation_status,
            "feasibility_status": feasibility_status,
            "primary_blocker": primary_blocker,
            "recommended_next_step": recommended_next_step,
        }
    ]
)
display(feasibility_decision.T)


,0
dataset_scope,Current DOT API extract
agency,DOT
target_name,missed_resolution_target
required_target_timestamps,"created_date, closed_date, due_date"
total_record_count,1469836
due_date_non_null_count,0
due_date_coverage_pct,0.0
closed_date_non_null_count,1437880
closed_date_coverage_pct,97.82588
eligible_record_count,0


**Interpretation.** `NOT_FEASIBLE` is produced only when the computed eligible
count is zero. A non-zero count yields `REQUIRES_FURTHER_REVIEW`, not an
automatic production approval.


## 29. Export of analytical outputs


In [28]:
exports = {
    "target_required_columns.csv": required_column_check,
    "target_timestamp_parse_summary.csv": timestamp_parse_summary,
    "target_field_coverage.csv": target_field_coverage,
    "target_eligibility_summary.csv": eligibility_summary,
    "target_ineligibility_reasons.csv": ineligibility_reason_summary,
    "target_coverage_by_complaint_type.csv": coverage_by_complaint_type,
    "target_coverage_by_status.csv": coverage_by_status,
    "target_coverage_by_year.csv": coverage_by_year,
    "target_coverage_by_agency.csv": coverage_by_agency,
    "target_timestamp_checks.csv": timestamp_checks,
    "target_alternative_assessment.csv": alternative_target_assessment,
    "target_leakage_assessment.csv": leakage_assessment,
    "target_feasibility_decision.csv": feasibility_decision,
}
for filename, frame in exports.items():
    frame.to_csv(REPORT_TABLES_DIR / filename, index=False)


def markdown_table(frame: pd.DataFrame) -> str:
    """Render a dataframe as dependency-free Markdown."""
    display_frame = frame.copy().where(pd.notna(frame), "")
    headers = [str(column) for column in display_frame.columns]

    def clean(value: object) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(map(clean, headers)) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    lines.extend(
        "| " + " | ".join(clean(value) for value in row) + " |"
        for row in display_frame.itertuples(index=False, name=None)
    )
    return "\n".join(lines)


target_definition_draft = f"""# Target Definition Draft

Generated by `notebooks/03_target_feasibility.ipynb` at
{extraction_completed_at_utc.isoformat()}.

## 1. Purpose

Record the candidate target definition and its current evidence-based
feasibility status. This is a draft because the modelling scope and final
eligibility rules are not approved.

## 2. Target name

`{TARGET_NAME}`

## 3. Candidate business definition

`1` means a complaint closed after its expected due date. `0` means it
closed on or before its expected due date.

## 4. Candidate formula

```python
eligible_for_target = (
    created_date.notna()
    & closed_date.notna()
    & due_date.notna()
)
missed_resolution_target = (closed_date > due_date).astype("int8")
```

## 5. Required fields

`unique_key`, `created_date`, `closed_date`, `due_date`, and `status`.

## 6. Eligible population

Historical records containing usable creation, closure, and expected due
timestamps. The target is constructed only inside this population.

## 7. Current DOT dataset scope

Official NYC Open Data dataset `{DATASET_ID}`, agency `{AGENCY}`, from
`{CONFIGURED_START_DATE}` through snapshot maximum
`{snapshot_maximum_created_date}`. The full paginated extract contained
{len(df):,} records.

## 8. Current target-field coverage

{markdown_table(target_field_coverage)}

## 9. Current eligible-record count

{eligible_record_count:,} records ({eligible_record_percentage:.6f}%).

## 10. Open complaint treatment

Open complaints without a closure timestamp cannot receive a historical
closed-on-time versus closed-late label under this definition. They are not
relabelled from status.

## 11. Cancelled and duplicate complaint considerations

Cancelled and duplicate complaints require a documented eligibility rule
before final target creation. Final treatment cannot be approved until a
dataset with usable due dates is selected.

## 12. Timestamp assumptions

Parsed timestamps are timezone-aware UTC values. Source nulls remain null,
parsing failures are reported separately, and no timestamp is imputed.
`due_date` is assumed to represent an expected deadline only provisionally;
its business semantics still require confirmation in a viable source scope.

## 13. Prediction moment

Shortly after complaint creation.

## 14. Label-construction fields

`created_date` defines context and eligibility. `closed_date` supplies the
actual outcome timestamp. `due_date` supplies the expected boundary.

## 15. Leakage exclusions

`closed_date`, final `status`, `resolution_description`, and
`resolution_action_updated_date` must not be creation-time model features.
`due_date` feature use is conditional on proving that it exists at the
prediction moment.

## 16. Invalid substitute targets

{markdown_table(alternative_target_assessment)}

## 17. Current feasibility decision

**{feasibility_status}** — {feasibility_reason}

Target creation status: **{target_creation_status}**.

## 18. Primary blocker

{primary_blocker}.

## 19. Known limitations

The NYC Open Data source is live, so future runs can retrieve a different
snapshot and must recompute every count. A non-zero eligible population
would still require due-date semantic validation, class-balance analysis,
status eligibility rules, and prediction-time availability review.

## 20. Required next investigation

{recommended_next_step}.
"""
TARGET_DEFINITION_DRAFT_PATH.write_text(
    target_definition_draft.strip() + "\n",
    encoding="utf-8",
)

exported_outputs = pd.DataFrame(
    {
        "output_path": [
            str((REPORT_TABLES_DIR / filename).relative_to(PROJECT_ROOT))
            for filename in exports
        ]
        + [str(TARGET_DEFINITION_DRAFT_PATH.relative_to(PROJECT_ROOT))]
    }
)
missing_outputs = [
    path
    for path in exported_outputs["output_path"]
    if not (PROJECT_ROOT / path).is_file()
]
if missing_outputs:
    raise RuntimeError(f"Required outputs were not created: {missing_outputs}")
display(exported_outputs)


,output_path
0,reports/tables/target_required_columns.csv
1,reports/tables/target_timestamp_parse_summary.csv
2,reports/tables/target_field_coverage.csv
3,reports/tables/target_eligibility_summary.csv
4,reports/tables/target_ineligibility_reasons.csv
5,reports/tables/target_coverage_by_complaint_type.csv
6,reports/tables/target_coverage_by_status.csv
7,reports/tables/target_coverage_by_year.csv
8,reports/tables/target_coverage_by_agency.csv
9,reports/tables/target_timestamp_checks.csv


## 30. Final conclusion


In [29]:
modelling_may_proceed = feasibility_status != "NOT_FEASIBLE"
final_conclusion = f"""## Final Decision

The current DOT extract is **{feasibility_status.replace("_", " ")}** for
constructing `{TARGET_NAME}`.

The target requires usable `closed_date` and `due_date` values on the same
historically valid record. The full extract contains {len(df):,} records,
{closed_date_non_null_count:,} usable closure timestamps
({closed_date_coverage_pct:.6f}%), and {due_date_non_null_count:,} usable due
dates ({due_date_coverage_pct:.6f}%). Consequently,
{eligible_record_count:,} records ({eligible_record_percentage:.6f}%) meet
the minimum target-eligibility rule, and target creation is
**{target_creation_status}**.

Closed status, a missing closure timestamp, an arbitrary duration threshold,
and post-resolution fields are not valid substitutes: they change the
defined outcome or introduce future information.

**Model training {'may proceed only after the documented further review' if modelling_may_proceed else 'must not proceed'} using this DOT extract for the selected target.**
"""
display(Markdown(final_conclusion))


## Final Decision

The current DOT extract is **NOT FEASIBLE** for
constructing `missed_resolution_target`.

The target requires usable `closed_date` and `due_date` values on the same
historically valid record. The full extract contains 1,469,836 records,
1,437,880 usable closure timestamps
(97.825880%), and 0 usable due
dates (0.000000%). Consequently,
0 records (0.000000%) meet
the minimum target-eligibility rule, and target creation is
**not_created**.

Closed status, a missing closure timestamp, an arbitrary duration threshold,
and post-resolution fields are not valid substitutes: they change the
defined outcome or introduce future information.

**Model training must not proceed using this DOT extract for the selected target.**


## 31. Recommended next step


In [30]:
display(
    Markdown(
        f"**Next step:** {recommended_next_step}. "
        "That comparison belongs to the next scope-selection task and is "
        "not performed here."
    )
)


**Next step:** Compare target-field coverage across alternative agencies and complaint types. That comparison belongs to the next scope-selection task and is not performed here.